### Read & Write en Delta Lake
1. Escribir datos en Delta Lake(Managed Table)
2. Escribir datos en Delta Lake(External Table)

##### Se puede ver la documentación en https://docs.delta.io/delta-batch/

In [0]:
%run "../includes/configuration"

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS movie_demo
MANAGED LOCATION "abfss://demo@moviee.dfs.core.windows.net"


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType, DateType

In [0]:
movie_schema = StructType(fields = [
    StructField("movieId", IntegerType(), False),
    StructField("title", StringType(), True),
    StructField("budget", DoubleType(), True),
    StructField("homePage", StringType(), True),
    StructField("overview", StringType(), True),
    StructField("popularity", DoubleType(), True),
    StructField("yearReleaseDate", IntegerType(), True),
    StructField("ReleaseDate", DateType(), True),
    StructField("revenue", DoubleType(), True),
    StructField("durationTime", IntegerType(), True),
    StructField("movieStatus", StringType(), True),
    StructField("tagline", StringType(), True),
    StructField("voteAverage", DoubleType(), True),
    StructField("voteCount", IntegerType(), True)
])

In [0]:
movie_df = spark.read \
           .option("header",True) \
           .schema(movie_schema) \
           .csv(f"{bronze_inc_folder_path}/2024-12-30/movie.csv")

#### Escribir datos en Delta Lake(Managed Table)

In [0]:
movie_df.write.mode("overwrite").format("delta").saveAsTable("movie_demo.movies_managed")




In [0]:
%sql
SELECT * FROM movie_demo.movies_managed

#### Escribir datos en Delta Lake(External Table)

In [0]:
movie_df.write.mode("overwrite").format("delta").save("abfss://demo@moviee.dfs.core.windows.net/movies_external")






In [0]:
%sql
CREATE TABLE movie_demo.movies_external
USING DELTA
LOCATION "abfss://demo@moviee.dfs.core.windows.net/movies_external"

In [0]:
%sql
SELECT * FROM movie_demo.movies_external

#### Leer datos de una carpeta delta y guardarlos en un dataframe

In [0]:
movies_external_df = spark.read.format("delta").load("abfss://demo@moviee.dfs.core.windows.net/movies_external")

In [0]:
display(movies_external_df)

In [0]:
movie_df.write.mode("overwrite").format("delta").partitionBy("yearReleaseDate").saveAsTable("movie_demo.movies_partitioned")

In [0]:
%sql
SHOW PARTITIONS movie_demo.movies_partitioned

### Update & Delete en Delta Lake
1. Update desde Delta Lake
2. Delete desde Delta Lake

##### Se puede ver la documentación en https://docs.delta.io/delta-update/


In [0]:
%sql
SELECT * FROM movie_demo.movies_managed

#### Update en SQL

In [0]:
%sql
UPDATE movie_demo.movies_managed
SET durationTime = 60
WHERE yearReleaseDate = 2012

In [0]:
%sql
SELECT * FROM movie_demo.movies_managed WHERE yearReleaseDate = 2012

#### Update en Python

In [0]:
from delta.tables import *

deltaTable = DeltaTable.forName(spark, 'movie_demo.movies_managed')

# Declare the predicate by using a SQL-formatted string.
deltaTable.update(
  condition = "yearReleaseDate = 2013",
  set = { "durationTime": "100" }
)





In [0]:
%sql
SELECT * FROM movie_demo.movies_managed WHERE yearReleaseDate = 2013

#### Delete en SQL

In [0]:
%sql
DELETE from movie_demo.movies_managed
WHERE yearReleaseDate = 2014

In [0]:
%sql
SELECT * from movie_demo.movies_managed
WHERE yearReleaseDate = 2014

In [0]:
from delta.tables import *

deltaTable = DeltaTable.forName(spark, 'movie_demo.movies_managed')

# Declare the predicate by using a SQL-formatted string.
deltaTable.delete("yearReleaseDate = 2015")







In [0]:
%sql
SELECT * from movie_demo.movies_managed
WHERE yearReleaseDate = 2015

### Merge/Upsert en Delta Lake
##### Se puede ver la documentación en https://docs.delta.io/delta-update/

#### Para ver esto, primero me voy a crear 3 dataframes distintos y guardarlos en vistas temporales

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType, DateType

In [0]:
movie_schema = StructType(fields = [
    StructField("movieId", IntegerType(), False),
    StructField("title", StringType(), True),
    StructField("budget", DoubleType(), True),
    StructField("homePage", StringType(), True),
    StructField("overview", StringType(), True),
    StructField("popularity", DoubleType(), True),
    StructField("yearReleaseDate", IntegerType(), True),
    StructField("ReleaseDate", DateType(), True),
    StructField("revenue", DoubleType(), True),
    StructField("durationTime", IntegerType(), True),
    StructField("movieStatus", StringType(), True),
    StructField("tagline", StringType(), True),
    StructField("voteAverage", DoubleType(), True),
    StructField("voteCount", IntegerType(), True)
])

In [0]:
movie_day1_df = spark.read \
           .option("header",True) \
           .schema(movie_schema) \
           .csv(f"{bronze_inc_folder_path}/2024-12-30/movie.csv") \
           .filter("yearReleaseDate < 2000") \
           .select("movieId","title","yearReleaseDate","releaseDate","durationTime")

In [0]:
movie_day1_df.createOrReplaceTempView("movie_day1")

In [0]:
from pyspark.sql.functions import upper

movie_day2_df = spark.read \
           .option("header",True) \
           .schema(movie_schema) \
           .csv(f"{bronze_inc_folder_path}/2024-12-30/movie.csv") \
           .filter("yearReleaseDate BETWEEN 1998 AND 2005") \
           .select("movieId",upper("title").alias("title"),"yearReleaseDate","releaseDate","durationTime")

In [0]:
movie_day2_df.createOrReplaceTempView("movie_day2")

In [0]:
from pyspark.sql.functions import upper

movie_day3_df = spark.read \
           .option("header",True) \
           .schema(movie_schema) \
           .csv(f"{bronze_inc_folder_path}/2024-12-30/movie.csv") \
           .filter("yearReleaseDate BETWEEN 1983 AND 1998 OR yearReleaseDate BETWEEN 2006 AND 2010") \
           .select("movieId",upper("title").alias("title"),"yearReleaseDate","releaseDate","durationTime")

In [0]:
movie_day3_df.createOrReplaceTempView("movie_day3")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS movie_demo.movies_merge(
  movieId INT,
  title STRING,
  yearReleaseDate INT,
  releaseDate DATE,
  durationTime INT,
  createdDate DATE,
  updatedDate DATE
)


#### Dia 1 (Merge en SQL)

In [0]:
%sql
MERGE INTO movie_demo.movies_merge mm
USING movie_day1 md1
ON mm.movieId = md1.movieId
WHEN MATCHED THEN
  UPDATE SET
    title = md1.title,
    yearReleaseDate = md1.yearReleaseDate,
    releaseDate = md1.releaseDate,
    durationTime = md1.durationTime,
    updatedDate = current_timestamp
WHEN NOT MATCHED
  THEN INSERT (movieId,title,yearReleaseDate,releaseDate,durationTime,createdDate)
  VALUES (md1.movieId,md1.title,md1.yearReleaseDate,md1.releaseDate,md1.durationTime,current_timestamp)

In [0]:
%sql
SELECT * FROM movie_demo.movies_merge

#### Dia 2 (Merge en SQL)

In [0]:
%sql
MERGE INTO movie_demo.movies_merge mm
USING movie_day2 md2
ON mm.movieId = md2.movieId
WHEN MATCHED THEN
  UPDATE SET
    title = md2.title,
    yearReleaseDate = md2.yearReleaseDate,
    releaseDate = md2.releaseDate,
    durationTime = md2.durationTime,
    updatedDate = current_timestamp
WHEN NOT MATCHED
  THEN INSERT (movieId,title,yearReleaseDate,releaseDate,durationTime,createdDate)
  VALUES (md2.movieId,md2.title,md2.yearReleaseDate,md2.releaseDate,md2.durationTime,current_timestamp)

In [0]:
%sql
SELECT * FROM movie_demo.movies_merge

#### Dia 3 (Merge en Python)

In [0]:
from delta.tables import *

deltaTablePeople = DeltaTable.forName(spark, 'movie_demo.movies_merge')

deltaTablePeople.alias('mm') \
  .merge(
    movie_day3_df.alias('md3'),
    'mm.movieId = md3.movieId'
  ) \
  .whenMatchedUpdate(set =
    {
      "title": "md3.title",
      "yearReleaseDate": "md3.yearReleaseDate",
      "releaseDate": "md3.releaseDate",
      "durationTime": "md3.durationTime",
      "updatedDate": "current_timestamp()"
    }
  ) \
  .whenNotMatchedInsert(values =
    {
      "movieId": "md3.movieId",
      "title": "md3.title",
      "yearReleaseDate": "md3.yearReleaseDate",
      "releaseDate": "md3.releaseDate",
      "durationTime": "md3.durationTime",
      "createdDate": "current_timestamp()"
    }
  ) \
  .execute()


In [0]:
%sql
SELECT * FROM movie_demo.movies_merge

### History, Time Travel y Vacuum
1. Historia y control de versiones
2. Viaje en el tiempo
3. Vacío


#### History y control de versiones

In [0]:
%sql
DESC HISTORY movie_demo.movies_merge

#### Viaje en el tiempo
##### Esto es volver a obtener la info que tenía la tabla en una versión o momento

In [0]:
%sql
SELECT * FROM movie_demo.movies_merge VERSION AS OF 1

In [0]:
%sql
SELECT * FROM movie_demo.movies_merge TIMESTAMP AS OF '2026-03-22T16:06:19.000+00:00'

In [0]:
df = spark.read.format("delta").option("timestampAsOf", "2026-03-22T16:06:19.000+00:00").table("movie_demo.movies_merge")







In [0]:
display(df)

#### Vacuum
##### Esto es eliminar el acceso a la historia de una tabla, ya que hay algunas empresas que por políticas, seguridad, regulaciones gubernamentales, etc, no quieren que se acceda a versiones viejas

In [0]:
%sql
VACUUM movie_demo.movies_merge

In [0]:
%sql
SELECT * FROM movie_demo.movies_merge TIMESTAMP AS OF '2026-03-22T16:06:19.000+00:00'

In [0]:
%sql
SET spark.databricks.delta.retentionDurationCheck.enabled = false;
VACUUM movie_demo.movies_merge RETAIN 0 HOURS;

In [0]:
%sql
SELECT * FROM movie_demo.movies_merge TIMESTAMP AS OF '2026-03-22T16:06:19.000+00:00'

In [0]:
%sql
DESC HISTORY movie_demo.movies_merge

#### Recuperar registros eliminados en una versión anterior

In [0]:
%sql
DELETE FROM movie_demo.movies_merge WHERE yearReleaseDate = 2004

In [0]:
%sql
DESC HISTORY movie_demo.movies_merge

In [0]:
%sql
MERGE INTO movie_demo.movies_merge mm
USING movie_demo.movies_merge VERSION AS OF 9 mmm
ON mm.movieId = mmm.movieId
WHEN NOT MATCHED THEN 
INSERT *

### Transaction Log en Delta Lake

In [0]:
%sql
CREATE TABLE IF NOT EXISTS movie_demo.movies_log(
  movieId INT,
  title STRING,
  yearReleaseDate INT,
  releaseDate DATE,
  durationTime INT,
  createdDate DATE,
  updatedDate DATE
)
USING DELTA

In [0]:
%sql
DESC EXTENDED movie_demo.movies_log

#### Vamos a hacer distintas operaciones para generar logs los cuales vamos a ver desde el Microsoft Azure Storage Explorer, en el word del resumen del curso voy explicando mejor como se ven los logs

In [0]:
%sql
INSERT INTO movie_demo.movies_log
SELECT * FROM movie_demo.movies_merge
WHERE movieId = 125537


In [0]:
%sql
INSERT INTO movie_demo.movies_log
SELECT * FROM movie_demo.movies_merge
WHERE movieId = 133575

In [0]:
%sql
DELETE FROM movie_demo.movies_log WHERE movieId = 133575

In [0]:
list = [118452,124606,125052,125123,125263,125537,126141,133575,142132,146269,157185]
for movieId in list:
    spark.sql(f"""INSERT INTO movie_demo.movies_log
              SELECT * FROM movie_demo.movies_merge
              WHERE movieId = {movieId}""")
    


In [0]:
%sql
INSERT INTO movie_demo.movies_log
SELECT * FROM movie_demo.movies_merge

### Convertir formato "Parquet" a "Delta"

#### Convertir tabla parquet a delta

In [0]:
%sql
CREATE TABLE IF NOT EXISTS movie_demo.movies_convert_to_delta(
  movieId INT,
  title STRING,
  yearReleaseDate INT,
  releaseDate DATE,
  durationTime INT,
  createdDate DATE,
  updatedDate DATE
)
USING PARQUET
LOCATION 'abfss://demo@moviee.dfs.core.windows.net/movies_convert_to_delta'

In [0]:
%sql
INSERT INTO movie_demo.movies_convert_to_delta
SELECT * FROM movie_demo.movies_merge

In [0]:
%sql
DESC HISTORY movie_demo.movies_convert_to_delta

In [0]:
%sql
CONVERT TO DELTA movie_demo.movies_convert_to_delta

In [0]:
%sql
DESC HISTORY movie_demo.movies_convert_to_delta

#### Convertir directorio parquet a delta

In [0]:
df = spark.table("movie_demo.movies_convert_to_delta")



In [0]:
df.write.mode("overwrite").format("parquet").save("abfss://demo@moviee.dfs.core.windows.net/movies_convert_to_delta_new")

In [0]:
%sql
CONVERT TO DELTA parquet.`abfss://demo@moviee.dfs.core.windows.net/movies_convert_to_delta_new`